In [1]:
import os
import PyPDF2
import numpy as np
import pandas as pd
from tabula.io import read_pdf
from datetime import datetime
import re

In [2]:
file_name = r"C:\Users\admin\Downloads\29.11.2023 £801.62 Amber Performance.pdf"

r"C:\Users\admin\Downloads\29.11.2023 £801.62 Amber Performance.pdf"

'C:\\Users\\admin\\Downloads\\29.11.2023 £801.62 Amber Performance.pdf'

In [3]:
invoice_type = "Products"

input_file = fr"C:\Users\admin\Downloads\29.11.2023 £801.62 Amber Performance.pdf"

In [4]:
table1 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(171, 344, 268, 562),
                  columns=[453, 562],
                  pandas_options={'header': None},
                  encoding="windows-1254")

heading = table1[0]
display(heading)

name = "Amber Performance"
docnum = heading[1][0]
print(docnum)

date = heading[1][1]
date = str(datetime.strptime(date, "%d/%m/%Y"))
print(date)

ordernum, transfernum = None, None
print(ordernum)
print(transfernum)


,0,1
0,Invoice No.,488036
1,Invoice/Tax Date,29/11/2023
2,Cust. Order No.,NaN
3,Account No.,ML


488036
2023-11-29 00:00:00
None
None


In [5]:
table2 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(319, 23.8, 665.9, 560),
                  columns=[74.8, 298, 347.4, 407.1, 462.2, 510, 560],
                  pandas_options={'header': None},
                  encoding="windows-1254")
content=table2[0]

content

,0,1,2,3,4,5,6
0,1.0,MMFT-SW-10-90 MISHIMOTO 10AN 90 DEGRE,9.73,2.92,6.81,20.0,1.36
1,1.0,MMTC-F80-15 - MISHIMOTO TRANSMISSION C,354.13,106.24,247.89,20.0,49.58
2,1.0,MMOC-E60-06 MISHIMOTO OIL COOLER,497.76,149.33,348.43,20.0,69.69
3,2.0,MMFT-SW-10-ST MISHIMOTO -10AN BRAIDED,7.57,4.54,10.60,20.0,2.12
4,1.0,MMTL-ANWR-10 AN FITTING WRENCH,7.57,2.27,5.30,20.0,1.06
5,1.0,MMSBH-10120-CB -10 BRAIDED LINE - BLACK,44.33,13.30,31.03,20.0,6.21


In [6]:
content.rename(columns={
    0: 'Quantity',
    1: 'Details',
    2: 'Unit Price',
    3: 'Disc Amount',
    4: 'Net Amount',
    5: 'VAT%',
    6: 'VAT'}, inplace=True)

display(content)
print(content)

,Quantity,Details,Unit Price,Disc Amount,Net Amount,VAT%,VAT
0,1.0,MMFT-SW-10-90 MISHIMOTO 10AN 90 DEGRE,9.73,2.92,6.81,20.0,1.36
1,1.0,MMTC-F80-15 - MISHIMOTO TRANSMISSION C,354.13,106.24,247.89,20.0,49.58
2,1.0,MMOC-E60-06 MISHIMOTO OIL COOLER,497.76,149.33,348.43,20.0,69.69
3,2.0,MMFT-SW-10-ST MISHIMOTO -10AN BRAIDED,7.57,4.54,10.60,20.0,2.12
4,1.0,MMTL-ANWR-10 AN FITTING WRENCH,7.57,2.27,5.30,20.0,1.06
5,1.0,MMSBH-10120-CB -10 BRAIDED LINE - BLACK,44.33,13.30,31.03,20.0,6.21


   Quantity                                  Details  Unit Price  Disc Amount  \
0       1.0    MMFT-SW-10-90 MISHIMOTO 10AN 90 DEGRE        9.73         2.92   
1       1.0   MMTC-F80-15 - MISHIMOTO TRANSMISSION C      354.13       106.24   
2       1.0         MMOC-E60-06 MISHIMOTO OIL COOLER      497.76       149.33   
3       2.0    MMFT-SW-10-ST MISHIMOTO -10AN BRAIDED        7.57         4.54   
4       1.0           MMTL-ANWR-10 AN FITTING WRENCH        7.57         2.27   
5       1.0  MMSBH-10120-CB -10 BRAIDED LINE - BLACK       44.33        13.30   

   Net Amount  VAT%    VAT  
0        6.81  20.0   1.36  
1      247.89  20.0  49.58  
2      348.43  20.0  69.69  
3       10.60  20.0   2.12  
4        5.30  20.0   1.06  
5       31.03  20.0   6.21  


In [7]:
#Spliting Details to SKU and Description
content[['SKU','Description']] = content['Details'].str.split(' ', n=1, expand=True)
display(content)

,Quantity,Details,Unit Price,Disc Amount,Net Amount,VAT%,VAT,SKU,Description
0,1.0,MMFT-SW-10-90 MISHIMOTO 10AN 90 DEGRE,9.73,2.92,6.81,20.0,1.36,MMFT-SW-10-90,MISHIMOTO 10AN 90 DEGRE
1,1.0,MMTC-F80-15 - MISHIMOTO TRANSMISSION C,354.13,106.24,247.89,20.0,49.58,MMTC-F80-15,- MISHIMOTO TRANSMISSION C
2,1.0,MMOC-E60-06 MISHIMOTO OIL COOLER,497.76,149.33,348.43,20.0,69.69,MMOC-E60-06,MISHIMOTO OIL COOLER
3,2.0,MMFT-SW-10-ST MISHIMOTO -10AN BRAIDED,7.57,4.54,10.60,20.0,2.12,MMFT-SW-10-ST,MISHIMOTO -10AN BRAIDED
4,1.0,MMTL-ANWR-10 AN FITTING WRENCH,7.57,2.27,5.30,20.0,1.06,MMTL-ANWR-10,AN FITTING WRENCH
5,1.0,MMSBH-10120-CB -10 BRAIDED LINE - BLACK,44.33,13.30,31.03,20.0,6.21,MMSBH-10120-CB,-10 BRAIDED LINE - BLACK


In [8]:
dict_content = content.to_dict(orient='records')
dict_content

line_items=[]
for item in dict_content:
    # print(item)
    partNum = item['SKU']
    desc = item['Description']
    quantity = item['Quantity']
    netTotal = item['Net Amount']

    print(partNum)

    line_item = {"line_type":"inventory",
                "sku": partNum,
                "name": desc,
                "Quantity": int(quantity),
                "net_total": float(netTotal),
                "tax_type": "INPUT2"}
    
    line_items.append(line_item)

print(line_items)

MMFT-SW-10-90
MMTC-F80-15
MMOC-E60-06
MMFT-SW-10-ST
MMTL-ANWR-10
MMSBH-10120-CB
[{'line_type': 'inventory', 'sku': 'MMFT-SW-10-90', 'name': 'MISHIMOTO 10AN 90 DEGRE', 'Quantity': 1, 'net_total': 6.81, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMTC-F80-15', 'name': '- MISHIMOTO TRANSMISSION C', 'Quantity': 1, 'net_total': 247.89, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMOC-E60-06', 'name': 'MISHIMOTO OIL COOLER', 'Quantity': 1, 'net_total': 348.43, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMFT-SW-10-ST', 'name': 'MISHIMOTO -10AN BRAIDED', 'Quantity': 2, 'net_total': 10.6, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMTL-ANWR-10', 'name': 'AN FITTING WRENCH', 'Quantity': 1, 'net_total': 5.3, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMSBH-10120-CB', 'name': '-10 BRAIDED LINE - BLACK', 'Quantity': 1, 'net_total': 31.03, 'tax_type': 'INPUT2'}]


In [9]:
table3 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(689, 345, 784, 560),
                  columns=[453, 560],
                  pandas_options={'header': None},
                  encoding='windows-1254')

total_content=table3[0]
display(total_content)


,0,1
0,Total Net Amount,650.06
1,Carriage Net,17.95
2,Total VAT Amount,133.61
3,Invoice Total,801.62


In [10]:
row_index1 = total_content.index[total_content[0] == 'Carriage Net'].tolist()[0]
row_index2 = total_content.index[total_content[0] == 'Invoice Total'].tolist()[0]

Shipping = float(total_content[1][row_index1])
final_total = float(total_content[1][row_index2])
display(Shipping)
display(final_total)

17.95

801.62

In [11]:
#Adding shipping to the line_items
line_shipping = {"line_type":"shipping_expense",
            "sku": None,
            "name": "shipping",
            "Quantity": int(1),
            "net_total": float(Shipping),
            "tax_type": "INPUT2"}

line_items.append(line_shipping)
print(line_items)

[{'line_type': 'inventory', 'sku': 'MMFT-SW-10-90', 'name': 'MISHIMOTO 10AN 90 DEGRE', 'Quantity': 1, 'net_total': 6.81, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMTC-F80-15', 'name': '- MISHIMOTO TRANSMISSION C', 'Quantity': 1, 'net_total': 247.89, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMOC-E60-06', 'name': 'MISHIMOTO OIL COOLER', 'Quantity': 1, 'net_total': 348.43, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMFT-SW-10-ST', 'name': 'MISHIMOTO -10AN BRAIDED', 'Quantity': 2, 'net_total': 10.6, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMTL-ANWR-10', 'name': 'AN FITTING WRENCH', 'Quantity': 1, 'net_total': 5.3, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMSBH-10120-CB', 'name': '-10 BRAIDED LINE - BLACK', 'Quantity': 1, 'net_total': 31.03, 'tax_type': 'INPUT2'}, {'line_type': 'Postage, Freight & Courier', 'sku': None, 'name': 'shipping', 'Quantity': 1, 'net_total': 17.95, 'tax_type': 'INPUT2'}]


In [12]:
payload = {}
keys = ["Source File",
        "Type",
        "Name",
        "Date",
        "Reference No.",
        "Order No.",
        "Transfer No.",
        "Document No.",
        "Line Items",
        "Total"]

values = [file_name,
        invoice_type,
        name,
        date,
        docnum,
        ordernum,
        transfernum,
        None,
        line_items,
        final_total]

for i, key in enumerate(keys):
    payload[key] = values[i]

payload

{'Source File': 'C:\\Users\\admin\\Downloads\\29.11.2023 £801.62 Amber Performance.pdf',
 'Type': 'Products',
 'Name': 'Amber Performance',
 'Date': '2023-11-29 00:00:00',
 'Reference No.': '488036',
 'Order No.': None,
 'Transfer No.': None,
 'Document No.': None,
 'Line Items': [{'line_type': 'inventory',
   'sku': 'MMFT-SW-10-90',
   'name': 'MISHIMOTO 10AN 90 DEGRE',
   'Quantity': 1,
   'net_total': 6.81,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'MMTC-F80-15',
   'name': '- MISHIMOTO TRANSMISSION C',
   'Quantity': 1,
   'net_total': 247.89,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'MMOC-E60-06',
   'name': 'MISHIMOTO OIL COOLER',
   'Quantity': 1,
   'net_total': 348.43,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'MMFT-SW-10-ST',
   'name': 'MISHIMOTO -10AN BRAIDED',
   'Quantity': 2,
   'net_total': 10.6,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'MMTL-ANWR-10',
   'name': 'AN FITTING WR